<img src="https://raw.githubusercontent.com/ComplianceAnalytics/aml-book1/main/assets/cal_logo_banner.png" alt="Compliance Analytics Ltd" width="300" onerror="this.style.display='none'">

# Applied AML Analytics: Turning Data Science Skills into Compliance Decisions
## Chapter 5 — Entity Resolution
### Companion Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_05.ipynb)

---

**Book:** *Applied AML Analytics: Turning Data Science Skills into Compliance Decisions* — Book 1  
**Publisher companion repository:** [github.com/ComplianceAnalytics/aml-book1](https://github.com/ComplianceAnalytics/aml-book1)  
**Dataset:** Northgate Retail Bank (synthetic — all data is fictional)  
**Chapters covered:** 3 · 4 · **5 (this notebook)** · 6 · 7 · 8 · 9  

> **How to use this notebook**  
> Run cells top-to-bottom using **Shift+Enter** or the ▶ button. The setup cell (Section 0) must run first — it generates the Northgate dataset that all later cells depend on. You do not need to install anything; all required libraries are pre-installed in Google Colab.

---

## Contents

| Section | Description | Exercise link |
|---------|-------------|---------------|
| **0. Setup** | Generate the Northgate ER dataset | — |
| **1. The identity problem** | Six mule accounts — what exact matching sees | Exercise 5.1 |
| **2. Fuzzy name matching** | Jaro-Winkler similarity across all account name pairs | Exercise 5.2 |
| **3. Multi-field matching** | Combine name, address, and phone evidence | Exercise 5.3 |
| **4. Entity network** | Build and visualise the entity graph with networkx | Exercise 5.3 |


---
## Section 0 — Setup

Run this cell first. It installs `rapidfuzz` (for fuzzy matching) and `networkx` (for graph construction), then generates the extended Northgate ER dataset — the standard 500-account population with deliberate name variants and shared attributes added to the six mule accounts.

In [ ]:
# Install libraries (Colab has these but rapidfuzz needs explicit install)
!pip install rapidfuzz --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from rapidfuzz import fuzz
import itertools

np.random.seed(42)
n = 500

# --- Standard Northgate population (494 non-mule accounts) ---
cash_in   = np.concatenate([np.random.lognormal(6.0,0.5,350), np.random.lognormal(8.2,0.4,100), np.random.lognormal(7.5,0.6,44)])
txn_count = np.maximum(np.concatenate([np.random.poisson(2.1,350), np.random.poisson(8.5,100), np.random.poisson(3.5,44)]),1).astype(float)
age       = np.concatenate([np.random.randint(12,120,350), np.random.randint(6,60,100), np.random.randint(24,84,44)])
ids_base  = [f'NRB_{str(i+1).zfill(3)}' for i in range(494)]

df_base = pd.DataFrame({
    'account_id':           ids_base,
    'account_name':         [f'Customer_{i}' for i in range(494)],
    'address':              [f'{np.random.randint(1,200)} High Street, London' for _ in range(494)],
    'phone_number':         [f'077{np.random.randint(10,99)} {np.random.randint(100000,999999)}' for _ in range(494)],
    'account_open_date':    pd.date_range('2022-01-01', periods=494, freq='3D').strftime('%Y-%m-%d').tolist(),
    'avg_monthly_cash_in':  np.round(cash_in,2),
    'avg_transaction_count':np.round(txn_count,1),
    'account_age_months':   age,
    'is_sar_worthy':        0,
})

# --- Six mule accounts with deliberate name variants and shared attributes ---
mule_data = [
    ('NRB_495','James Okafor',       '14 Millbank Court, London E4 7RQ', '07700 900421', '2024-01-12', 8800, 4.2),
    ('NRB_496','J. Okafor',          '14 Millbank Court, London E4 7RQ', '07700 900421', '2024-01-15', 9100, 3.9),
    ('NRB_497','Jimmy Okafor',       '14 Millbank Ct, London E4 7RQ',    '07700 900 421','2024-01-18', 8650, 4.5),
    ('NRB_498','Blessing Okafor',    '14 Millbank Court, E4 7RQ',        '07700 900422', '2024-01-19', 9200, 4.1),
    ('NRB_499','B. Okafor-Williams', '14 Millbank Court, London',        '',             '2024-01-22', 8950, 3.8),
    ('NRB_500','Adaeze Williams',    'Flat 2, 14 Millbank Court, London E4','07700 900422','2024-01-25',9050, 4.3),
]
df_mule = pd.DataFrame(mule_data, columns=[
    'account_id','account_name','address','phone_number','account_open_date',
    'avg_monthly_cash_in','avg_transaction_count'])
df_mule['account_age_months'] = 16
df_mule['is_sar_worthy'] = 1

df = pd.concat([df_base, df_mule], ignore_index=True)
print(f'Dataset ready: {len(df)} accounts ({df.is_sar_worthy.sum()} mule accounts)')
df[df.is_sar_worthy==1][['account_id','account_name','address','phone_number','account_open_date']]


---
## Section 1 — The Identity Problem (Exercise 5.1)

How many of the six mule accounts produce an **exact name match** with any other account?
This section shows what an account-level system without entity resolution sees.

In [ ]:
mule_names = set(df[df.is_sar_worthy==1]['account_name'])

# Exact matches only
exact_matches = []
for name in mule_names:
    matches = df[df['account_name'] == name]
    if len(matches) > 1:
        exact_matches.append(name)

print(f'Mule account names: {sorted(mule_names)}')
print(f'Exact name matches across the full dataset: {len(exact_matches)}')
print()
print('Six alerts as seen by the TMS — no entity connections visible:')
df[df.is_sar_worthy==1][['account_id','account_name','avg_monthly_cash_in']].to_string(index=False) |> print


---
## Section 2 — Fuzzy Name Matching (Exercise 5.2)

Apply **Jaro-Winkler similarity** across all account pairs. Candidate pairs above the 0.75 threshold are surfaced.
Observe which mule accounts connect — and which one does **not** appear in any high-similarity pair.

In [ ]:
# Compute pairwise name similarity for mule accounts vs all accounts
mule_ids = df[df.is_sar_worthy==1]['account_id'].tolist()

pairs = []
for i, row_a in df[df.is_sar_worthy==1].iterrows():
    for j, row_b in df.iterrows():
        if row_a['account_id'] >= row_b['account_id']:
            continue
        score = fuzz.token_sort_ratio(row_a['account_name'], row_b['account_name']) / 100
        if score >= 0.75:
            pairs.append({
                'account_a': row_a['account_id'], 'name_a': row_a['account_name'],
                'account_b': row_b['account_id'], 'name_b': row_b['account_name'],
                'name_similarity': round(score, 3),
                'b_is_mule': int(row_b['is_sar_worthy'])
            })

candidates = pd.DataFrame(pairs).sort_values('name_similarity', ascending=False)
print(f'Candidate pairs (name similarity ≥ 0.75): {len(candidates)}')
print()
print('Mule-to-mule matches:')
print(candidates[candidates.b_is_mule==1].to_string(index=False))
print()
unmatched = set(mule_ids) - set(candidates['account_a']) - set(candidates['account_b'])
print(f'Mule accounts NOT appearing in any name-similarity pair: {unmatched}')


---
## Section 3 — Multi-Field Matching (Exercise 5.3)

Combine **name similarity + address overlap + shared phone number** into a single evidence score.
The account that eluded name matching should now be connected.

In [ ]:
def address_overlap(a, b):
    """Fraction of words in common between two address strings."""
    wa = set(str(a).lower().split())
    wb = set(str(b).lower().split())
    if not wa or not wb: return 0.0
    return len(wa & wb) / max(len(wa), len(wb))

def phone_match(a, b):
    """1 if phones match after stripping spaces, 0 otherwise."""
    a_clean = str(a).replace(' ','').replace('-','')
    b_clean = str(b).replace(' ','').replace('-','')
    return 1.0 if a_clean and b_clean and a_clean == b_clean else 0.0

mule_df = df[df.is_sar_worthy==1].reset_index(drop=True)
multi_pairs = []

for i, row_a in mule_df.iterrows():
    for j, row_b in df.iterrows():
        if row_a['account_id'] >= row_b['account_id']:
            continue
        name_s  = fuzz.token_sort_ratio(row_a['account_name'], row_b['account_name']) / 100
        addr_s  = address_overlap(row_a['address'], row_b['address'])
        phone_s = phone_match(row_a['phone_number'], row_b['phone_number'])
        combined = 0.4*name_s + 0.4*addr_s + 0.2*phone_s
        if combined >= 0.5 or phone_s == 1.0 or addr_s >= 0.6:
            multi_pairs.append({
                'account_a': row_a['account_id'], 'account_b': row_b['account_id'],
                'name_sim': round(name_s,3), 'addr_sim': round(addr_s,3),
                'phone_match': int(phone_s), 'combined': round(combined,3),
                'b_is_mule': int(row_b['is_sar_worthy'])
            })

confirmed = pd.DataFrame(multi_pairs).sort_values('combined', ascending=False)
mule_links = confirmed[confirmed.b_is_mule==1]
print(f'Multi-field confirmed mule-to-mule links: {len(mule_links)}')
print(mule_links.to_string(index=False))


---
## Section 4 — Entity Network (Exercise 5.3 continued)

Build the entity graph and visualise the connected components.
All six mule accounts should appear in the **same connected component**.

In [ ]:
G = nx.Graph()

# Add all accounts as nodes
for _, row in df.iterrows():
    G.add_node(row['account_id'], name=row['account_name'], is_mule=int(row['is_sar_worthy']))

# Add edges from multi-field confirmed pairs
for _, pair in confirmed.iterrows():
    G.add_edge(pair['account_a'], pair['account_b'], weight=pair['combined'])

# Add edges for shared phone numbers (exact)
phone_groups = df[df['phone_number']!=''].groupby('phone_number')['account_id'].apply(list)
for phone, accs in phone_groups.items():
    if len(accs) > 1:
        for a, b in itertools.combinations(accs, 2):
            G.add_edge(a, b, edge_type='shared_phone')

# Report connected components
components = list(nx.connected_components(G))
print(f'Total connected components: {len(components)}')
for i, comp in enumerate(sorted(components, key=len, reverse=True)[:5]):
    mule_ct = sum(1 for n in comp if G.nodes[n]['is_mule'])
    if mule_ct > 0:
        print(f'  Component {i+1}: {len(comp)} accounts, {mule_ct} mule accounts → HIGH PRIORITY')

# Visualise the mule-containing component
mule_component = next(c for c in components if any(G.nodes[n]['is_mule'] for n in c))
subG = G.subgraph(mule_component)
colours = ['#e74c3c' if G.nodes[n]['is_mule'] else '#3498db' for n in subG.nodes()]
labels  = {n: G.nodes[n]['name'].split()[0] for n in subG.nodes()}

plt.figure(figsize=(10, 7))
pos = nx.spring_layout(subG, seed=42)
nx.draw_networkx(subG, pos, node_color=colours, labels=labels,
                 node_size=800, font_size=8, font_color='white',
                 edge_color='#95a5a6', width=1.5)
plt.title('Northgate Entity Network — Mule-Containing Component\n(red = mule account, blue = linked account)', pad=15)
plt.axis('off')
plt.tight_layout()
plt.savefig('entity_network.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Network saved. Accounts in mule component: {len(mule_component)}')


---
## What's Next

In Chapter 6, you will group the 500 Northgate accounts into behavioural segments using K-Means clustering. You will discover that the mule accounts — now identified as a single network — form a distinct cluster with unusually high cash-in amounts relative to transaction frequency. Segmentation is the foundation for making rule thresholds segment-aware rather than applying the same threshold to every customer.

Open Chapter 6: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ComplianceAnalytics/aml-book1/blob/main/notebooks/chapter_06.ipynb)